In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.9 MB/s eta 0:00:00


In [ ]:
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API")
# wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 2310080019 (lkx100-kl-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import os
from pathlib import Path
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
import wandb

In [ ]:
RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:



class PestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        # Skip hidden folders starting with . or ._
        self.classes = sorted([d.name for d in self.root_dir.iterdir()
                               if d.is_dir() and not d.name.startswith('.')])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        self.samples = []
        skipped_count = 0
        for cls_name in self.classes:
            cls_dir = self.root_dir / cls_name
            for img_path in cls_dir.glob('*'):
                # Skip hidden files starting with . or ._
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    self.samples.append((str(img_path), self.class_to_idx[cls_name]))

        if skipped_count > 0:
            print(f'[INFO] Skipped {skipped_count} excluded images')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# YOLOv8-CLS Wrapper -----------------------

class YOLOv8Classifier(nn.Module):
    def __init__(self, num_classes=3, model_size='n', pretrained=True):

        super(YOLOv8Classifier, self).__init__()
        if pretrained:
            model_name = f'yolov8{model_size}-cls.pt'
            yolo_wrapper = YOLO(model_name)
            print(f"✓ Loaded pretrained YOLOv8{model_size}-cls model")
        else:
            model_name = f'yolov8{model_size}-cls.yaml'
            yolo_wrapper = YOLO(model_name)
            print(f"✓ Loaded YOLOv8{model_size}-cls architecture (no pretrained weights)")
        self.model = yolo_wrapper.model
        self.num_classes = num_classes
        replaced = False
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear) and module.out_features == 1000:
                parts = name.split('.')
                parent = self.model
                for part in parts[:-1]:
                    if part.isdigit():
                        parent = parent[int(part)]
                    else:
                        parent = getattr(parent, part)

                attr_name = parts[-1]
                in_features = module.in_features

                # Replace the layer
                if attr_name.isdigit():
                    parent[int(attr_name)] = nn.Linear(in_features, num_classes)
                else:
                    setattr(parent, attr_name, nn.Linear(in_features, num_classes))

                print(f"✓ Replaced classifier '{name}': {in_features} -> {num_classes} classes")
                replaced = True
                break

        if not replaced:
            print("⚠️ Warning: Could not find 1000-class classifier layer to replace")
            print("   All Linear layers found:")
            for name, module in self.model.named_modules():
                if isinstance(module, nn.Linear):
                    print(f"     - {name}: Linear({module.in_features} -> {module.out_features})")

    def forward(self, x):
        """
        Forward pass through YOLOv8-CLS model
        Returns logits for classification
        """
        # YOLOv8-cls model may return (logits, features) tuple
        output = self.model(x)

        # If output is a tuple, extract just the logits (first element)
        if isinstance(output, tuple):
            return output[0]
        return output

    def unfreeze_backbone(self):
        """Unfreeze all parameters for fine-tuning"""
        for param in self.model.parameters():
            param.requires_grad = True
        print("✓ Unfroze all model parameters for fine-tuning")

In [ ]:
#Utilities -----------------------

def calculate_class_weights(dataset, indices):
    class_counts = {}
    for idx in indices:
        _, label = dataset.samples[idx]
        class_counts[label] = class_counts.get(label, 0) + 1
    total_samples = len(indices)
    num_classes = len(class_counts)
    weights = [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)]
    return torch.FloatTensor(weights)


def calculate_additional_metrics(y_true, y_pred, y_prob, num_classes):
    """Calculate precision, recall, f1, and roc_auc"""
    metrics = {}

    # Macro averaged metrics
    metrics['precision'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # ROC AUC
    try:
        if num_classes == 2:
            metrics['roc_auc'] = roc_auc_score(y_true, y_prob[:, 1])
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metrics['roc_auc'] = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
    except:
        metrics['roc_auc'] = 0.0

    return metrics


#  Early Stopping -----------------------
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_val_acc = 0

    def __call__(self, val_acc):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
            self.best_val_acc = val_acc
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_val_acc = val_acc
            self.counter = 0


#Plot Confusion Matrix -----------------------
def plot_cm(cm, class_names, title):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)

    img_path = f"{title.replace(' ', '_')}.png"
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()
    return img_path

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, num_classes):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []

    pbar = tqdm(dataloader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

        pbar.set_postfix({'loss': running_loss/len(dataloader), 'acc': 100.*correct/total})

    acc = 100. * correct / total
    avg_loss = running_loss / len(dataloader)
    extra_metrics = calculate_additional_metrics(all_labels, all_preds, np.array(all_probs), num_classes)
    return avg_loss, acc, extra_metrics, all_labels, all_preds, all_probs


In [ ]:
def validate(model, dataloader, criterion, device, num_classes):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation', leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            pbar.set_postfix({'loss': running_loss/len(dataloader), 'acc': 100.*correct/total})

    acc = 100. * correct / total
    avg_loss = running_loss / len(dataloader)

    extra_metrics = calculate_additional_metrics(all_labels, all_preds, np.array(all_probs), num_classes)

    return avg_loss, acc, extra_metrics, all_labels, all_preds, all_probs



In [ ]:
# Main -----------------------

def main():

    DATASET_PATH = "/content/drive/MyDrive/Dataset/castor_v2_224x224"
    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.0001
    N_SPLITS = 5
    PATIENCE = 9
    PROJECT_NAME = "YOLOv8-cls_KFold"
    GROUP_NAME = "Original-2"

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    # GPU info
    if torch.cuda.is_available():
        print(f'GPU Count: {torch.cuda.device_count()}')
        for i in range(torch.cuda.device_count()):
            print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'Current GPU: {torch.cuda.current_device()}')
    else:
        print('WARNING: CUDA not available, running on CPU!')

    transform = transforms.Compose([
        # transforms.Resize((640, 640)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print('Loading dataset...')
    full_dataset = PestDataset(DATASET_PATH, transform=transform)
    print(f'Total samples: {len(full_dataset)}')
    print(f'Classes: {full_dataset.classes}')

    # Print per-class sample count for verification
    class_counts = {}
    for _, label in full_dataset.samples:
        class_name = full_dataset.classes[label]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    print(f'Samples per class:')
    for cls_name, count in sorted(class_counts.items()):
        print(f'  - {cls_name}: {count}')

    # Prepare indices and labels for stratified k-fold
    indices = np.arange(len(full_dataset))
    labels = np.array([label for _, label in full_dataset.samples])

    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

    fold_results = []
    all_fold_data = []  # Store best metrics for final summary table

    print(f'\n{"="*60}')
    print(f'Starting {N_SPLITS}-Fold Cross-Validation')
    print(f'{"="*60}\n')

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        print(f'\n{"="*60}')
        print(f'FOLD {fold + 1}/{N_SPLITS}')
        print(f'{"="*60}')

        is_last_fold = (fold == N_SPLITS - 1)

        CONFIG={
                'fold': fold + 1,
                'batch_size': BATCH_SIZE,
                'epochs': NUM_EPOCHS,
                'learning_rate': LEARNING_RATE,
                'model': 'yolov8n-cls',  # Using pure classification model
                'patience': PATIENCE,
                'architecture': 'YOLOv8-CLS (pure)',
                'pretrained': True,
                'usewandb':False
        }
        # Initialize W&B for this fold
        run_name = f"{GROUP_NAME}_Fold-{fold + 1}"
        if CONFIG.get("usewandb"):
            wandb.init(
                project=PROJECT_NAME,
                name=run_name,
                group=GROUP_NAME,
                config=CONFIG,
                reinit=True
            )
            print(f"✓ W&B run initialized: {run_name}")
        else:
            print("W&B disabled for this fold; skipping init.")

        # Calculate class weights for this fold
        class_weights = calculate_class_weights(full_dataset, train_idx)
        print(f'Class weights for fold {fold + 1}: {class_weights}')

        # Create data loaders
        train_dataset = Subset(full_dataset, train_idx)
        val_dataset = Subset(full_dataset, val_idx)

        print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        # Initialize pure YOLOv8-CLS model
        model = YOLOv8Classifier(
            num_classes=len(full_dataset.classes),
            model_size='n',  # nano model for speed
            pretrained=True
        ).to(device)
        model.unfreeze_backbone()

        # Verify model is on GPU
        print(f'Model device: {next(model.parameters()).device}')

        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
        )

        # Early stopping
        early_stopping = EarlyStopping(patience=PATIENCE, verbose=True)

        best_val_acc = 0.0
        best_epoch = -1
        best_prec = best_rec = best_f1 = best_loss = best_roc = 0
        best_val_cm = None
        best_train_cm = None
        best_val_report = None
        best_train_report = None
        best_model_path = f'YOLOv8_{GROUP_NAME}_fold{fold+1}.pth'

        for epoch in range(NUM_EPOCHS):
            print(f'\nEpoch [{epoch+1}/{NUM_EPOCHS}]')

            train_loss, train_acc, train_metrics, train_labels, train_preds, train_probs = train_epoch(
                model, train_loader, criterion, optimizer, device, len(full_dataset.classes)
            )
            val_loss, val_acc, val_metrics, val_labels, val_preds, val_probs = validate(
                model, val_loader, criterion, device, len(full_dataset.classes)
            )

            print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
            print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

            # Step scheduler based on validation accuracy
            scheduler.step(val_acc)
            current_lr = optimizer.param_groups[0]['lr']
            print(f'Learning Rate: {current_lr:.6f}')

            # Log to W&B
            if CONFIG.get("usewandb"):
                wandb.log({
                    'epoch': epoch + 1,
                    'learning_rate': current_lr,
                    'train_loss': train_loss,
                    'train_accuracy': train_acc,
                    'train_precision': train_metrics['precision'],
                    'train_recall': train_metrics['recall'],
                    'train_f1': train_metrics['f1'],
                    'train_auc': train_metrics['roc_auc'],
                    'val_loss': val_loss,
                    'val_accuracy': val_acc,
                    'val_precision': val_metrics['precision'],
                    'val_recall': val_metrics['recall'],
                    'val_f1': val_metrics['f1'],
                    'val_auc': val_metrics['roc_auc']
                })
            else:
                # Minimal console summary when W&B is off
                print(f"[Epoch {epoch+1}] train_loss={train_loss:.4f} val_loss={val_loss:.4f} train_acc={train_acc:.2f}% val_acc={val_acc:.2f}%")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_prec = val_metrics['precision']
                best_rec = val_metrics['recall']
                best_f1 = val_metrics['f1']
                best_loss = val_loss
                best_roc = val_metrics['roc_auc']
                best_epoch = epoch + 1

                torch.save(model.state_dict(), best_model_path)
                if CONFIG.get("usewandb"):
                    wandb.save(best_model_path)
                else:
                    print(f"Saved best model locally to {best_model_path} (W&B disabled)")
                print(f'✓ Saved best model for fold {fold+1} (Val Acc: {val_acc:.2f}%)')

                # Store confusion matrices and classification reports
                best_val_cm = confusion_matrix(val_labels, val_preds)
                best_val_report = classification_report(val_labels, val_preds, target_names=full_dataset.classes, output_dict=False)
                best_train_cm = confusion_matrix(train_labels, train_preds)
                best_train_report = classification_report(train_labels, train_preds, target_names=full_dataset.classes, output_dict=False)

            # Early stopping check
            early_stopping(val_acc)
            if early_stopping.early_stop:
                print(f'\nEarly stopping triggered at epoch {epoch+1}')
                break

        # Log confusion matrices and classification reports
        if best_val_cm is not None:
            val_cm_img = plot_cm(best_val_cm, full_dataset.classes, f"Fold{fold+1}_Validation_CM")
            train_cm_img = plot_cm(best_train_cm, full_dataset.classes, f"Fold{fold+1}_Training_CM")

            # Print per-class validation accuracy for inspection
            per_class_acc = np.diag(best_val_cm) / (best_val_cm.sum(axis=1) + 1e-12)
            for i, cls_name in enumerate(full_dataset.classes):
                print(f"   Class '{cls_name}': val_acc = {per_class_acc[i]*100:.2f}% (samples = {best_val_cm.sum(axis=1)[i]})")

            if CONFIG.get("usewandb"):
                wandb.log({
                    "best_validation_confusion_matrix": wandb.Image(val_cm_img),
                    "best_training_confusion_matrix": wandb.Image(train_cm_img),
                    "best_validation_classification_report": wandb.Html(f"<pre>{best_val_report}</pre>"),
                    "best_training_classification_report": wandb.Html(f"<pre>{best_train_report}</pre>")
                })
            else:
                print("W&B disabled; skipping confusion matrix logging.")

        # Store this fold's metrics in summary
        if CONFIG.get("usewandb"):
            wandb.summary["fold"] = fold + 1
            wandb.summary["best_epoch"] = best_epoch
            wandb.summary["best_val_loss"] = best_loss
            wandb.summary["final_val_loss"] = best_loss
            wandb.summary["final_val_accuracy"] = best_val_acc
            wandb.summary["final_precision"] = best_prec
            wandb.summary["final_recall"] = best_rec
            wandb.summary["final_f1"] = best_f1
            wandb.summary["final_roc_auc"] = best_roc

        # Log summary table on last fold
        if is_last_fold:
            # Add current fold's data
            all_fold_data.append([
                fold + 1, best_epoch, best_val_acc, best_loss, best_prec, best_rec, best_f1, best_roc
            ])

            # Create summary table with ALL folds
            if CONFIG.get("usewandb"):
                summary_table = wandb.Table(
                    columns=["fold", "best_epoch", "best_accuracy", "best_loss", "best_precision", "best_recall", "best_f1", "best_roc_auc"]
                )
                for fold_data in all_fold_data:
                    summary_table.add_data(*fold_data)
                wandb.log({"all_folds_best_metrics": summary_table})
            else:
                print("W&B disabled; skipping all-folds summary upload.")

        fold_results.append({
            'fold': fold + 1,
            'best_val_acc': best_val_acc,
            'best_epoch': best_epoch,
            'best_loss': best_loss,
            'best_prec': best_prec,
            'best_rec': best_rec,
            'best_f1': best_f1,
            'best_roc': best_roc
        })

        # Store data for folds 1-4 (fold 5 adds itself inside the is_last_fold block)
        if not is_last_fold:
            all_fold_data.append([
                fold + 1, best_epoch, best_val_acc, best_loss, best_prec, best_rec, best_f1, best_roc
            ])

        print(f'\nFold {fold + 1} Best Validation Accuracy: {best_val_acc:.2f}%')
        print(f'{"="*60}')

        if CONFIG.get("usewandb"):
            wandb.finish()
        else:
            print("W&B disabled; nothing to finish for this fold.")

    # Summary
    print(f'\n{"="*60}')
    print('K-FOLD CROSS-VALIDATION SUMMARY')
    print(f'{"="*60}')

    for result in fold_results:
        print(f"Fold {result['fold']}: Best Val Acc = {result['best_val_acc']:.2f}% (epoch {result['best_epoch']})")

    # Calculate averages
    avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
    avg_loss = np.mean([r['best_loss'] for r in fold_results])
    avg_prec = np.mean([r['best_prec'] for r in fold_results])
    avg_rec = np.mean([r['best_rec'] for r in fold_results])
    avg_f1 = np.mean([r['best_f1'] for r in fold_results])
    avg_roc = np.mean([r['best_roc'] for r in fold_results])

    print(f"\n====== FINAL K-FOLD RESULTS (AVG OVER ALL FOLDS) ======")
    print(f"Avg Val Acc  : {avg_acc:.4f}")
    print(f"Avg Val Loss : {avg_loss:.4f}")
    print(f"Avg Precision: {avg_prec:.4f}")
    print(f"Avg Recall   : {avg_rec:.4f}")
    print(f"Avg F1 Score : {avg_f1:.4f}")
    print(f"Avg ROC AUC  : {avg_roc:.4f}")
    print(f'{"="*60}')


In [ ]:

if __name__ == '__main__':
    main()